## `PriorityClass`, preemption & how the scheduler decides

### PriorityClass and preemption

By default a Pending pod waits politely. A **`PriorityClass`** tags a workload as important enough that lower-priority pods should be kicked out for it:

```yaml
kind: PriorityClass
metadata: { name: high-priority }
value: 1000000
preemptionPolicy: PreemptLowerPriority   # or Never
---
spec:
  priorityClassName: high-priority
```

When a high-priority pod can't fit, the scheduler asks: is there a node where **evicting lower-priority pods** would make it fit? If so, it evicts them (they return to Pending) and schedules the important one. Two built-ins ship: `system-cluster-critical` and `system-node-critical` — reserved for CoreDNS, kube-proxy, CNI; don't use those values for apps. `preemptionPolicy: Never` respects priority for *queue order* but never preempts — good for batch work.

### How the scheduler decides — filter, score, profiles

The loop in detail:

- **Filter (predicates)** — each plugin returns yes/no; a node must pass *all*: `NodeResourcesFit`, `NodeAffinity`, `NodePorts`, `PodTopologySpread`, `TaintToleration`, `VolumeBinding`.
- **Score (priorities)** — each assigns 0–100, summed and weighted; highest wins: `NodeResourcesBalancedAllocation`, `NodeResourcesFit` (`LeastAllocated` default = pack lightly, or `MostAllocated` = bin-pack), `InterPodAffinity`, `ImageLocality` (prefer nodes with the image cached).
- **Profiles** — a named plugin config. Target one with `spec.schedulerName`, or run multiple schedulers.

You won't author a profile on the CKA, but know the field exists and that "the scheduler" isn't a monolith. This closes the module: requests/limits, affinity, taints, spread, priority — every lever over *where a Pod runs*. On our map it all resolves at the **kube-scheduler** in the control plane.